# Hackoweek Task 5 — Model Evaluation & Feature Engineering

## Objective
Build and evaluate a binary classification model using the Titanic dataset. The notebook demonstrates train/test split, cross-validation, confusion matrix, precision, recall, F1-score, ROC-AUC, feature engineering, feature scaling, and missing-data handling.

## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay, classification_report,
    precision_score, recall_score, f1_score, roc_auc_score, roc_curve
)

import warnings
warnings.filterwarnings('ignore')

## 2. Load the Dataset
The Titanic dataset is loaded directly from Seaborn, so no separate dataset file is required.

In [ ]:
df = sns.load_dataset('titanic')
df.head()

In [ ]:
print('Dataset Shape:', df.shape)
print('\nColumn Names:')
print(df.columns.tolist())

df.info()

## 3. Exploratory Data Check

In [ ]:
print('Missing values:')
print(df.isnull().sum()[df.isnull().sum() > 0])

plt.figure(figsize=(10, 5))
sns.heatmap(df.isnull(), cbar=False)
plt.title('Missing Values in Titanic Dataset')
plt.show()

## 4. Feature Engineering
Create `family_size` from siblings/spouses and parents/children. This gives the model a more meaningful representation of family size.

In [ ]:
df['family_size'] = df['sibsp'] + df['parch'] + 1
df[['sibsp', 'parch', 'family_size']].head()

## 5. Select Features and Target
`survived` is the target. We use a compact set of numerical and categorical predictors.

In [ ]:
features = ['pclass', 'sex', 'age', 'fare', 'family_size', 'embarked']
X = df[features]
y = df['survived']

print('Features:', features)
print('Target: survived')

## 6. Train/Test Split
The data is divided into training and testing sets. Stratification keeps the class proportions similar in both sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print('Training samples:', X_train.shape[0])
print('Testing samples:', X_test.shape[0])

## 7. Missing Data Handling and Feature Scaling
Missing numerical values are replaced with the median, while missing categorical values are replaced with the most frequent category. Numerical features are standardized. One-hot encoding converts categorical features into numerical columns.

Using a pipeline prevents data leakage because preprocessing is fitted only on the training data.

In [ ]:
numeric_features = ['pclass', 'age', 'fare', 'family_size']
categorical_features = ['sex', 'embarked']

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_features),
    ('cat', categorical_pipeline, categorical_features)
])

## 8. Build and Train the Logistic Regression Model

In [ ]:
model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print('Model training completed successfully.')

## 9. Cross-Validation
Five-fold stratified cross-validation is used to estimate model performance across multiple train/validation splits.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy')

print('Cross-validation accuracy scores:', np.round(cv_scores, 4))
print('Mean CV accuracy:', round(cv_scores.mean(), 4))
print('CV standard deviation:', round(cv_scores.std(), 4))

## 10. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Did Not Survive', 'Survived'])
disp.plot()
plt.title('Confusion Matrix')
plt.show()

print('Confusion Matrix:')
print(cm)

## 11. Precision, Recall and F1-Score

In [ ]:
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print('Precision:', round(precision, 4))
print('Recall:', round(recall, 4))
print('F1-score:', round(f1, 4))

print('\nClassification Report:')
print(classification_report(y_test, y_pred))

## 12. ROC Curve and ROC-AUC

In [ ]:
roc_auc = roc_auc_score(y_test, y_prob)
fpr, tpr, thresholds = roc_curve(y_test, y_prob)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f'Logistic Regression (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], linestyle='--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.show()

print('ROC-AUC:', round(roc_auc, 4))

## 13. Final Evaluation Summary

In [ ]:
accuracy = model.score(X_test, y_test)

results = pd.DataFrame({
    'Metric': ['Test Accuracy', 'Mean CV Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC'],
    'Score': [accuracy, cv_scores.mean(), precision, recall, f1, roc_auc]
})

results['Score'] = results['Score'].round(4)
results

## 14. Conclusion
The experiment demonstrates a complete classification workflow. Missing values are handled through imputation, a new `family_size` feature is engineered, numerical features are scaled, and categorical variables are encoded. The Logistic Regression model is evaluated using a held-out test set and five-fold cross-validation. Confusion matrix, precision, recall, F1-score, and ROC-AUC provide complementary views of classification performance.